In [1]:
import os
from typing import TypedDict,Union,Dict,List
from langchain_core.messages import HumanMessage,AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END  # noqa: F401
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
class AgentState(TypedDict):
    messages:List[Union[HumanMessage,AIMessage]]
    


In [4]:
llm = ChatOpenAI(model="gpt-5-nano", timeout=30, max_retries=1)

In [5]:
def process(state:AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    state["messages"].append(AIMessage(content=response.content))
    print(response.content)
    return state

In [6]:
graph=StateGraph(AgentState)
graph.add_node("process",process)
graph.add_edge(START,"process")
graph.add_edge("process",END)
app=graph.compile()

In [ ]:
conversation_history = []

while True:
    user_input = input("User (type 'exit' to stop): ").strip()
    if user_input.lower() in {"exit", "quit", "q"}:
        break
    if not user_input:
        continue

    conversation_history.append(HumanMessage(content=user_input))
    result = app.invoke({"messages": conversation_history})
    conversation_history = result["messages"]